# Family Matters: A Study of the Association between Public Assistance and Family Composition
### Ricardo Hernández, Ayush Bhat, Antonios Koskinas

#### Part 1: Data Cleaning

##### Import libraries

In [11]:
import pandas as pd
import statsmodels.formula.api as smf
import scipy.stats as st
import matplotlib.pyplot as plt
from statsmodels.iolib.summary2 import summary_col
import numpy as np

##### Import csv file (sourced from IPUMS USA)

In [2]:
df = pd.read_csv('/Users/ayushbhat/Desktop/ECO4421/sauce.csv.gz')

In [3]:
df.columns

Index(['YEAR', 'SAMPLE', 'SERIAL', 'CBSERIAL', 'HHWT', 'CLUSTER', 'STATEICP',
       'STRATA', 'GQ', 'PERNUM', 'PERWT', 'NCHILD', 'SEX', 'AGE', 'MARST',
       'RACE', 'RACED', 'HISPAN', 'HISPAND', 'HCOVANY', 'HINSCAID', 'EDUC',
       'EDUCD', 'INCTOT', 'FTOTINC', 'INCWELFR'],
      dtype='str')

##### Drop unused columns (Note: year dropped since sample is single year)

In [4]:
df = df.drop(columns=["YEAR", "SAMPLE", "CBSERIAL", "GQ", "RACED", "HISPAND", "EDUC"])

##### Filter the data by women

In [5]:
df = df[df["SEX"] == 2].copy()

##### Map and replace missing codes in INCTOT, FTOTINC and INCWELFR

In [6]:
missing_codes = {
    "INCTOT":   [9999998, 9999999],
    "FTOTINC":  [9999998, 9999999],
    "INCWELFR": [99999],
}

for col, codes in missing_codes.items():
    df[col] = df[col].replace(codes, np.nan)

##### Filter data by age (15-50)

In [7]:
df = df[df["AGE"].between(15, 50)].copy()

##### Map IPUMS USA state ICP to state names

In [8]:
icpnamemapping = {
    1: 'CONNECTICUT',
    2: 'MAINE',
    3: 'MASSACHUSETTS',
    4: 'NEWHAMPSHIRE',
    5: 'RHODEISLAND',
    6: 'VERMONT',
    11: 'DELAWARE',
    12: 'NEWJERSEY',
    13: 'NEWYORK',
    14: 'PENNSYLVANIA',
    21: 'ILLINOIS',
    22: 'INDIANA',
    23: 'MICHIGAN',
    24: 'OHIO',
    25: 'WISCONSIN',
    31: 'IOWA',
    32: 'KANSAS',
    33: 'MINNESOTA',
    34: 'MISSOURI',
    35: 'NEBRASKA',
    36: 'NORTHDAKOTA',
    37: 'SOUTHDAKOTA',
    40: 'VIRGINIA',
    41: 'ALABAMA',
    42: 'ARKANSAS',
    43: 'FLORIDA',
    44: 'GEORGIA',
    45: 'LOUISIANA',
    46: 'MISSISSIPPI',
    47: 'NORTHCAROLINA',
    48: 'SOUTHCAROLINA',
    49: 'TEXAS',
    51: 'KENTUCKY',
    52: 'MARYLAND',
    53: 'OKLAHOMA',
    54: 'TENNESSEE',
    56: 'WESTVIRGINIA',
    61: 'ARIZONA',
    62: 'COLORADO',
    63: 'IDAHO',
    64: 'MONTANA',
    65: 'NEVADA',
    66: 'NEWMEXICO',
    67: 'UTAH',
    68: 'WYOMING',
    71: 'CALIFORNIA',
    72: 'OREGON',
    73: 'WASHINGTON',
    81: 'ALASKA',
    82: 'HAWAII',
    98: 'DC'}

df['STATENAME'] = df['STATEICP'].map(icpnamemapping)

##### Create race dummy variables

In [9]:
df['BLACK'] = [1 if i==2 else 0 for i in df['RACE']]
df['AIAN'] = [1 if i==3 else 0 for i in df['RACE']]
df['API'] = [1 if i==4 or i==5 or i==6 else 0 for i in df['RACE']]
df['OTHER'] = [1 if i==7 else 0 for i in df['RACE']]
df['MULTR'] = [1 if i==8 or i==9 else 0 for i in df['RACE']]
df['HISP'] = [1 if i!=0 else 0 for i in df['HISPAN']]

##### Create dichotomous expansion state dummy variable

In [10]:
df['EXPANSION'] = [0 if i==25 or i==32 or i==41 or i==43 or i==44 or i==46 or i==48 or i==49 or i==54 or i==68 else 1 for i in df['STATEICP']]

##### Drop unused columns

In [12]:
df = df.drop(columns=['STATEICP', 'RACE', 'HISPAN'])

##### Create dichotomous welfare recipiency dummy variable using INCWELFR

In [13]:
df['WELFARE'] = np.where(
    df['INCWELFR'].isna(),
    np.nan,
    (df['INCWELFR'] > 0).astype(int)
)

##### Create personal income (excl. welfare) variable using INCTOT and INCWELFR

In [14]:
df['PRSNINC'] = df['INCTOT'] - df['INCWELFR']

##### Map medicaid codes, create interaction variable, rescale income to units of 10,000

In [18]:
medicaid_mapping = {1:0, 2:1}
df['MEDICAID'] = df['HINSCAID'].map(medicaid_mapping)
df['MEDFARE'] = df['MEDICAID'] * df['WELFARE']

df['FAMINC'] = df['FTOTINC'] - df['INCTOT']
df['PRSNINC'] = df['PRSNINC'] / 10000
df['FAMINC'] = df['FAMINC'] / 10000
df['PRSNINC2'] = df['PRSNINC'] ** 2
df['FAMINC2'] = df['FAMINC'] ** 2

df['AGE2'] = df['AGE'] ** 2

##### Create dummy variables for education level, marital status (dichotomous), and LARGEFAM for logit

In [ ]:
df['HIGHSCHOOL'] = [1 if i == 63 or i==64 else 0 for i in df['EDUCD']]
df['BACHELORS'] = [1 if i == 101 else 0 for i in df['EDUCD']]
df['ADVANCED'] = [1 if i>=114 else 0 for i in df['EDUCD']]

df['MARD'] = [1 if i==1 or i==2 or i==3 else 0 for i in df['MARST']]

df['LARGEFAM'] = [1 if i >= 3 else 0 for i in df['NCHILD']]

In [16]:
df.shape

(720668, 37)

In [17]:
df.columns

Index(['SERIAL', 'HHWT', 'CLUSTER', 'STRATA', 'PERNUM', 'PERWT', 'NCHILD',
       'SEX', 'AGE', 'MARST', 'HCOVANY', 'HINSCAID', 'EDUCD', 'INCTOT',
       'FTOTINC', 'INCWELFR', 'STATENAME', 'BLACK', 'AIAN', 'API', 'OTHER',
       'MULTR', 'HISP', 'EXPANSION', 'WELFARE', 'PRSNINC', 'MEDICAID',
       'MEDFARE', 'FAMINC', 'PRSNINC2', 'FAMINC2', 'AGE2', 'HIGHSCHOOL',
       'BACHELORS', 'ADVANCED', 'MARD', 'LARGEFAM'],
      dtype='str')

#### Part 2: Regressions & Models

##### a. Initial Model

##### b. National level Baseline Model

##### c. Expansion Model & One-Way State Fixed Effects Model

In [19]:
ols_exp = 'NCHILD ~ MEDICAID + WELFARE + MEDFARE + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + MARD + BLACK + AIAN + API + OTHER + MULTR + HISP + EXPANSION + HIGHSCHOOL + BACHELORS + ADVANCED'
ols_fe = 'NCHILD ~ MEDICAID + WELFARE + MEDFARE + AGE + AGE2 + PRSNINC + PRSNINC2 + FAMINC + FAMINC2 + MARD + BLACK + AIAN + API + OTHER + MULTR + HISP + C(STATENAME, Treatment("FLORIDA")) + HIGHSCHOOL + BACHELORS + ADVANCED'

In [20]:
reg_exp = smf.ols(ols_exp, data=df).fit()
reg_fe = smf.ols(ols_fe, data=df).fit()

In [21]:
results = summary_col(
    [reg_exp, reg_fe],
    model_names=['Expansion', 'One-Way State Fixed Effects'],
    regressor_order=['MEDICAID', 'WELFARE', 'MEDFARE', 'EXPANSION', 'AGE', 'AGE2', 'PRSNINC', 'PRSNINC2', 'FAMINC', 'FAMINC2'],
    stars=True,
    float_format='%0.5f',
    info_dict={
        'Observations': lambda x: f'{int(x.nobs):,}'
    },
    drop_omitted=True
)

print(results)


                 Expansion    One-Way State Fixed Effects
---------------------------------------------------------
MEDICAID       0.31545***     0.32748***                 
               (0.00349)      (0.00351)                  
WELFARE        -0.32724***    -0.30899***                
               (0.02196)      (0.02190)                  
MEDFARE        0.61130***     0.59881***                 
               (0.02447)      (0.02440)                  
EXPANSION      -0.02219***                               
               (0.00280)                                 
AGE            0.17777***     0.17901***                 
               (0.00093)      (0.00093)                  
AGE2           -0.00209***    -0.00211***                
               (0.00001)      (0.00001)                  
PRSNINC        -279.41180***  -265.16700***              
               (4.14101)      (4.14755)                  
PRSNINC2       43852.95426*** 42353.59722***             
             

In [22]:
print(f'EXPANSION T TEST', reg_exp.t_test("EXPANSION = 0"), sep='\n\n')

EXPANSION T TEST

                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0            -0.0222      0.003     -7.914      0.000      -0.028      -0.017


In [23]:
print(f'MEDICAID * WELFARE T TEST', reg_fe.t_test("MEDFARE = 0"), sep='\n\n')

MEDICAID * WELFARE T TEST

                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.5988      0.024     24.538      0.000       0.551       0.647


In [24]:
ftest = reg_fe.f_test("MEDICAID = 0, WELFARE = 0, MEDFARE = 0")
print(f'JOINT SIGNIFICANCE F TEST', ftest, sep='\n\n')

JOINT SIGNIFICANCE F TEST

<F test: F=3817.0254949172, p=0.0, df_denom=6.78e+05, df_num=3>


##### d. Logit Expansion Model & Logit One-Way State Fixed Effects Model